<font size="6" color='grey'><b>KI-Agenten. Verstehen. Anwenden. Gestalten.</b></font>

<font size="5" color='grey'><b>Projekt-Template A – Research-Team</b></font>

---

> **Vorlage für M23 Multi-Agent-Projekt**
> Suche relevante `# ⚠️ TODO`-Kommentare und passe sie an dein Thema an.

**Team-Struktur:**

| Rolle | Agent | Aufgabe |
|-------|-------|---------|
| 🎯 Koordination | **Supervisor** | Routing & Planung |
| 🔍 Recherche | **Web-Researcher** | Informationen beschaffen |
| 📊 Analyse | **Analyst** | Ergebnisse zusammenfassen |

In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
# ⚠️ TODO: Passe den Projektnamen an deinen Team-Namen an
os.environ["LANGSMITH_PROJECT"]  = "M23-Research-Team"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.10
langchain-chroma                         1.1.0
langchain-classic                        1.0.2
langchain-community                      0.4.1
langchain-core                           1.2.18
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.0.10
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.9

IP-Adresse: 34.75.137.9
Hostname: 9.137.75.34.bc.googleusercontent.com
Stadt: North Charleston
Region: South Carolina
Land: US
Koordinaten: 32.8546,-79.9748
Provider: AS396982 Google LLC
Postleitzahl: 29415
Zeitzone: America/New_York

## 1 | Übersicht & Lernziel
---

Dieses Template implementiert ein **Research-Team** bestehend aus:
- **Web-Researcher:** Beschafft Informationen zu einem Thema
- **Analyst:** Wertet die Recherche aus und erstellt eine strukturierte Zusammenfassung
- **Supervisor:** Koordiniert die Reihenfolge (Recherche → Analyse → FINISH)

**Typischer Workflow:**
```
START → Supervisor → Web-Researcher → Supervisor → Analyst → Supervisor → FINISH
```

In [2]:
#@title
#@markdown <p><font size="4" color='green'>flowchart: Research-Team-Architektur</font></p>
diagram = '''
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nRouting & Koordination"]

    SUP -->|"researcher"| R["🔍 Web-Researcher\nInformationen beschaffen"]
    SUP -->|"analyst"| A["📊 Analyst\nZusammenfassen & Strukturieren"]
    SUP -->|"FINISH"| END([END])

    R -->|"AIMessage\n(name=Researcher)"| SUP
    A -->|"AIMessage\n(name=Analyst)"| SUP

    R --- RT1["🔎 web_search"]
    R --- RT2["📌 key_facts"]
    A --- AT1["📋 structure_report"]
    A --- AT2["🔢 count_words"]

    style SUP fill:#FF9800,color:#fff
    style R   fill:#2196F3,color:#fff
    style A   fill:#4CAF50,color:#fff
    style RT1 fill:#E3F2FD
    style RT2 fill:#E3F2FD
    style AT1 fill:#E8F5E9
    style AT2 fill:#E8F5E9
'''
mermaid(diagram, width=780)

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent
from IPython.display import Image as IPImage

In [ ]:
# Worker-LLM: gpt-4o-mini (Inhalte generieren, schnell & günstig)
worker_llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

# Supervisor-LLM: o3 (Routing-Entscheidungen, starkes Reasoning)
# ⚠️ o3 akzeptiert KEINEN temperature-Parameter (API-Fehler)
from langchain_core.messages import SystemMessage as SM
import langchain
_base_supervisor = init_chat_model("openai:o3")
print("✅ LLMs initialisiert (worker: gpt-4o-mini | supervisor: o3)")

In [ ]:
# ── Web-Researcher Tools ───────────────────────────────────────────────────

@tool
def web_search(query: str) -> str:
    """Simuliert eine Web-Suche zu einem Thema.
    Gibt relevante Fakten und Informationen zurück.
    Nutze klare, spezifische Suchanfragen.
    """
    # ⚠️ TODO (Optional): Ersetze durch echte Suche, z.B. DuckDuckGo oder Tavily
    # from langchain_community.tools import DuckDuckGoSearchRun
    # return DuckDuckGoSearchRun().run(query)
    return (
        f"Suchergebnisse für '{query}':\n"
        f"- Wichtige Informationsquelle 1: Kernfakten zu {query}\n"
        f"- Wichtige Informationsquelle 2: Hintergründe und Kontext\n"
        f"- Wichtige Informationsquelle 3: Aktuelle Entwicklungen\n"
        "(Simulierte Ergebnisse – ersetze durch echte Suche)"
    )

@tool
def key_facts(thema: str) -> str:
    """Extrahiert die 3–5 wichtigsten Fakten zu einem Thema.
    Nutze nach web_search zur Verdichtung der Ergebnisse.
    """
    return (
        f"Kernfakten zu '{thema}':\n"
        "1. Erster wichtiger Fakt\n"
        "2. Zweiter wichtiger Fakt\n"
        "3. Dritter wichtiger Fakt\n"
        "(Bitte auf Basis der Suchergebnisse konkretisieren)"
    )

# ── Analyst Tools ──────────────────────────────────────────────────────────

@tool
def structure_report(thema: str) -> str:
    """Erstellt eine Berichts-Gliederung für ein Thema.
    Nutze am Anfang um die Analyse zu strukturieren.
    """
    return (
        f"Berichtsstruktur für '{thema}':\n"
        "1. Executive Summary (2–3 Sätze)\n"
        "2. Hintergrund & Kontext\n"
        "3. Kernbefunde (3–5 Punkte)\n"
        "4. Bewertung & Implikationen\n"
        "5. Empfehlungen"
    )

@tool
def count_words(text: str) -> str:
    """Zählt Wörter in einem Text – nützlich um Längenvorgaben einzuhalten."""
    anzahl = len(text.split())
    return f"Der Text hat {anzahl} Wörter."

print("✅ Tools definiert: web_search, key_facts, structure_report, count_words")

In [ ]:
# ── Worker Agents ──────────────────────────────────────────────────────────

# ⚠️ TODO: Passe die System-Prompts an dein Recherche-Thema an

researcher_agent = create_react_agent(
    worker_llm,
    tools=[web_search, key_facts],
    prompt=SystemMessage(
        # ⚠️ TODO: Beschreibe hier das Spezialgebiet des Researchers
        "Du bist ein erfahrener Researcher. "
        "Nutze web_search um Informationen zu beschaffen und "
        "key_facts um die wichtigsten Punkte herauszuarbeiten. "
        "Sei präzise und faktenorientiert. Antworte auf Deutsch."
    )
)

analyst_agent = create_react_agent(
    worker_llm,
    tools=[structure_report, count_words],
    prompt=SystemMessage(
        # ⚠️ TODO: Beschreibe hier den Analyse-Fokus
        "Du bist ein Analyst. Nutze die Recherche-Ergebnisse des Researchers "
        "und erstelle mit structure_report einen klar gegliederten Bericht. "
        "Halte dich an Fakten aus der Recherche. Antworte auf Deutsch."
    )
)

print("✅ Worker Agents erstellt:")
print("   researcher_agent → [web_search, key_facts]")
print("   analyst_agent    → [structure_report, count_words]")

In [ ]:
# ── State Definition ───────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages:    Annotated[list, add_messages]  # Gesamte Nachrichten-Historie
    aufgabe:     str    # Original-Aufgabe (unveränderlich)
    naechster:   str    # Routing-Entscheidung des Supervisors
    iterationen: int    # Zähler gegen Endlosschleifen
    max_iter:    int    # Konfigurierbare Grenze (Standard: 6)

# ── Routing-Entscheidung mit Begründung ────────────────────────────────────
class SupervisorEntscheidung(BaseModel):
    """Strukturierte Routing-Entscheidung des Supervisors."""
    naechster: Literal["researcher", "analyst", "FINISH"] = Field(
        description="Nächster Agent oder FINISH wenn die Aufgabe vollständig ist."
    )
    begruendung: str = Field(
        description="Kurze Begründung für diese Entscheidung (1–2 Sätze)."
    )

supervisor_llm = (
    _base_supervisor
    .with_structured_output(SupervisorEntscheidung)
    .with_retry(stop_after_attempt=3)
)
print("✅ State und Supervisor-LLM konfiguriert")

In [ ]:
# ── Supervisor-Prompt ──────────────────────────────────────────────────────

SUPERVISOR_PROMPT = (
    "Du koordinierst ein Research-Team mit zwei Experten.\n\n"
    "<Team>\n"
    "- researcher: Beschafft Informationen und Fakten (IMMER zuerst)\n"
    "- analyst:    Wertet Recherche aus und schreibt Bericht (NACH researcher)\n"
    "</Team>\n\n"
    # ⚠️ TODO: Passe den Workflow an dein Projekt an
    "<Workflow>\n"
    "Standard-Reihenfolge: researcher → analyst → FINISH\n"
    "Wie du die Historie liest:\n"
    "  name=Researcher → researcher war aktiv\n"
    "  name=Analyst    → analyst war aktiv\n"
    "</Workflow>\n\n"
    "<Rules>\n"
    "1. researcher ZUERST – ohne Fakten kann der Analyst nicht arbeiten.\n"
    "2. analyst NUR EINMAL – nach der Recherche.\n"
    "3. FINISH sobald der Analyst seinen Bericht geliefert hat.\n"
    "4. Jeden Agenten maximal EINMAL aufrufen.\n"
    "</Rules>"
)
print("✅ Supervisor-Prompt konfiguriert")

In [ ]:
# ── Supervisor Node ────────────────────────────────────────────────────────
def supervisor_node(state: AgentState) -> dict:
    """Liest Situation, entscheidet wer als Nächstes arbeitet."""
    # Iterations-Schutz: verhindert Endlosschleifen
    if state["iterationen"] >= state["max_iter"]:
        mprint(f"⚠️ **Iterations-Limit ({state['max_iter']}) erreicht – FINISH erzwungen.**")
        return {"naechster": "FINISH"}

    entscheidung = supervisor_llm.invoke([
        SystemMessage(SUPERVISOR_PROMPT),
        HumanMessage(f"Aufgabe: {state['aufgabe']}"),
        *state["messages"],
    ])
    mprint(f"**👔 Supervisor → `{entscheidung.naechster}`** – _{entscheidung.begruendung}_")
    return {"naechster": entscheidung.naechster}


# ── Worker Node Factory ────────────────────────────────────────────────────
def make_worker_node(agent, name: str):
    """Erzeugt einen Worker-Node mit Fehlerbehandlung und Kontext-Übergabe."""
    def worker_node(state: AgentState) -> dict:
        mprint(f"\n**🤖 {name}** arbeitet...")
        # Aufgabe + bisherige Ergebnisse als Kontext übergeben
        kontext = f"Aufgabe: {state['aufgabe']}"
        if state["messages"]:
            bisherige = "\n".join(
                f"{getattr(m, 'name', 'User')}: {m.content[:300]}"
                for m in state["messages"]
            )
            kontext += f"\n\nBisherige Arbeit des Teams:\n{bisherige}"

        try:
            result = agent.invoke(
                {"messages": [HumanMessage(kontext)]},
                config={"recursion_limit": 15}
            )
            antwort = result["messages"][-1].content
        except Exception as e:
            # ✅ MVP-Anforderung: Mind. ein Fehlerpfad abgefangen
            antwort = f"❌ Fehler: {e} – bitte alternativen Weg wählen."
            mprint(f"  ⚠️ {name} fehlgeschlagen: {e}")

        return {
            "messages":    [AIMessage(content=antwort, name=name)],
            "iterationen": state["iterationen"] + 1,
        }
    worker_node.__name__ = f"{name}_node"
    return worker_node


# ── Router ─────────────────────────────────────────────────────────────────
def supervisor_router(state: AgentState) -> str:
    naechster = state.get("naechster", "FINISH")
    return "__end__" if naechster == "FINISH" else naechster

print("✅ Supervisor-Node, Worker-Nodes und Router definiert")

In [ ]:
# ── Graph aufbauen ────────────────────────────────────────────────────────
builder = StateGraph(AgentState)

# Nodes
builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", make_worker_node(researcher_agent, "Researcher"))
builder.add_node("analyst",    make_worker_node(analyst_agent,     "Analyst"))

# Einstiegspunkt
builder.add_edge(START, "supervisor")

# Bedingtes Routing
builder.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "researcher": "researcher",
        "analyst":    "analyst",
        "__end__": END,
    }
)

# Zurück zum Supervisor nach jedem Worker
builder.add_edge("researcher", "supervisor")
builder.add_edge("analyst",    "supervisor")

graph = builder.compile()
print("✅ Graph kompiliert")

# Visualisierung
try:
    display(IPImage(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"(Visualisierung nicht verfügbar: {e})")

In [ ]:
# ── Testlauf ──────────────────────────────────────────────────────────────
# ⚠️ TODO: Passe die Testaufgabe an dein Team-Thema an
testaufgabe = "Recherchiere die Grundprinzipien von LangGraph und erstelle einen strukturierten Überblick."

result = graph.invoke(
    {
        "messages":    [],
        "aufgabe":     testaufgabe,
        "naechster":   "",
        "iterationen": 0,
        "max_iter":    6,      # ✅ MVP-Schutz: max. 6 Worker-Calls
    },
    config={
        "run_name": "M23-Research-Team-Test",
        "tags":     ["m23-research"],
        "recursion_limit": 40,  # Äußere Grenze für den gesamten Graph
    }
)

mprint("## 🎯 Ergebnis")
mprint(f"**Aufgabe:** {testaufgabe}\n")
mprint(f"**Iterationen:** {result['iterationen']} von max. {result['max_iter']}\n---")
for msg in result["messages"]:
    agent = getattr(msg, "name", None) or "System"
    mprint(f"**[{agent}]**\n{msg.content}\n---")

In [ ]:
#@title 📊 LangSmith Trace{ display-mode: "form" }
import time; time.sleep(2)
show_trace("M23-Research-Team", limit=2, show_steps=True)

## ✅ MVP-Checkliste (vor Abgabe prüfen)

Dein Projekt ist **fertig**, wenn alle Punkte erfüllt sind:

| Kriterium | Geprüft? |
|-----------|----------|
| Supervisor routet zu **mindestens 2 Workern** | ☐ |
| Worker geben **sinnvolle Antworten** | ☐ |
| Mind. **ein Fehlerpfad** ist abgefangen (bereits im Template ✅) | ☐ |
| **End-to-End-Flow** funktioniert (1 Durchlauf ohne Fehler) | ☐ |
| **LangSmith-Trace** zeigt den Ablauf | ☐ |

**Nicht erforderlich:**
- ❌ Perfekte Ausgabe
- ❌ Schöne UI
- ❌ Mehr als 2 Worker

## A | Deine Aufgabe – Research-Team anpassen

---

### Was du bauen sollst

Du nimmst dieses Template und entwickelst daraus ein **konkretes Research-System** für ein selbst gewähltes Thema. Am Ende des Tages präsentierst du deinen Mitstreitern ein funktionierendes System, das eine echte Recherche-Aufgabe löst.

---

### Schritt-für-Schritt-Anleitung

#### Schritt 1 – Thema wählen (5 Min)

Einige sich im Team auf ein Recherche-Thema. Gute Beispiele:

| Thema | Beispiel-Aufgabe |
|-------|-----------------|
| 🤖 KI & Technologie | „Vergleiche GPT-4 und Claude 3 nach Stärken und Schwächen" |
| 🌍 Wirtschaft | „Analysiere die wichtigsten Risiken für den Euro in 2026" |
| 🔬 Wissenschaft | „Erkläre Quantencomputing und seine Praxisrelevanz" |
| 🏢 Euer Berufsfeld | Ein Thema aus eurem Arbeitsalltag |

Tragt euer Thema hier ein: **Unser Thema: ___________________________**

---

#### Schritt 2 – `web_search`-Tool anpassen (10 Min)

Das aktuelle Tool gibt simulierte Ergebnisse zurück. Du hast zwei Optionen:

**Option A – Simulation verfeinern** (einfacher, kein Extra-Key nötig):
```python
@tool
def web_search(query: str) -> str:
    """..."""
    # Ersetze den Platzhalter-Text durch themenspezifische Fakten
    return (
        f"Recherche zu '{query}':\n"
        "- Fakt 1: ...\n"
        "- Fakt 2: ...\n"
    )
```

**Option B – Echte Web-Suche** (realistischer, braucht Tavily-Key):
```python
from langchain_community.tools.tavily_search import TavilySearchResults
tavily = TavilySearchResults(max_results=3)

@tool
def web_search(query: str) -> str:
    """Sucht aktuelle Informationen im Web via Tavily."""
    results = tavily.invoke({"query": query})
    return "\n".join(r["content"] for r in results)
```

---

#### Schritt 3 – System-Prompts anpassen (10 Min)

Öffne die **Worker Agents**-Zelle und passe die `# ⚠️ TODO`-Kommentare an:

```python
# Researcher – WAS soll er recherchieren?
"Du bist Experte für [DEIN THEMA]. Recherchiere ..."

# Analyst – WELCHEN Fokus soll der Bericht haben?
"Du bist Analyst für [DEIN BEREICH]. Erstelle einen Bericht der ..."
```

---

#### Schritt 4 – Supervisor-Prompt anpassen (5 Min)

Öffne die **Supervisor-Prompt**-Zelle und prüfe:
- Ist die Reihenfolge `researcher → analyst → FINISH` für dein Thema richtig?
- Braucht dein System eine andere Logik? (z.B. Analyst zuerst?)

---

#### Schritt 5 – Testen & LangSmith prüfen (10 Min)

1. Passe `testaufgabe` in der Testlauf-Zelle auf dein Thema an
2. Führe alle Zellen von oben nach unten aus
3. Prüfe in LangSmith: Welcher Agent hat was gemacht?
4. Hake die MVP-Checkliste ab

---

### Mini-Demo am Ende (3 Min pro Team)

Zeigt euren Mitstreitern:
1. **Eine Beispiel-Anfrage** → Welche Antwort liefert das System?
2. **LangSmith-Trace** → Wer hat was entschieden?
3. **Eine Besonderheit** eures Systems (besonderes Tool, interessantes Routing, …)

---

### Bonus-Aufgaben (wenn das MVP läuft)

**Bonus 1 – Drittes Tool hinzufügen**
Ergänze z.B. ein `cite_sources`-Tool beim Researcher, das Quellen auflistet.

**Bonus 2 – Critique-Schleife**
Füge einen dritten `critic_agent` hinzu, der den Analyst-Bericht bewertet.
Wenn die Bewertung < 7/10: Supervisor schickt Analyst nochmals.

**Bonus 3 – Echte Persistenz**
Speichere den finalen Bericht als `.txt`-Datei in Google Drive.

```python
@tool
def save_report(inhalt: str, dateiname: str) -> str:
    """Speichert den fertigen Bericht in Google Drive."""
    pfad = f"/content/drive/MyDrive/{dateiname}.txt"
    with open(pfad, "w") as f:
        f.write(inhalt)
    return f"Bericht gespeichert: {pfad}"
```

**Bonus 4 – Human in the Loop**
Baue einen Genehmigungsschritt ein: Der Researcher-Bericht muss vom Trainer
freigegeben werden, bevor der Analyst weitermacht (→ *M17 Human-in-the-Loop*).
